# Road-accident models: MLP, GraphSAGE, GatedGCN and Graph Transformer

Select **Runtime > Change runtime type > GPU**, then run the cells in order.

This branch combines `baseline-fw`'s GPU embedding trainer and richer cached features with the shared edge-aware models, three-seed runner and full-network evaluation. It is a **modified reproduction**, not a strict recreation of the paper.

Data and outputs persist on Google Drive. The setup cell installs the single `requirements.txt` in a Python 3.12 environment. Start with Delaware (`STATE = "DE"`).

This copy adds residual edge-aware **GatedGCN** using native PyTorch. Inspection/attention visualizations are deferred. Push the implementation to the GitHub branch selected below before running in Colab.

This copy adds a **sparse edge-aware Graph Transformer**. Its attention follows incoming road connections, using road attributes in query-key scores, with residual attention/feed-forward blocks. It retains the shared per-road prediction head.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Private repositories require a token with access to this repository.
# Paste it into the hidden prompt, or leave blank for a public repository.
import base64
import getpass
import os
import subprocess
from pathlib import Path

REPO = "https://github.com/aaronteork/CS5284_GML.git"
BRANCH = "main"  # Set to the pushed branch containing Graph Transformer.
REPO_DIR = Path("/content/CS5284_GML_graph_transformer")

def prepare_repository():
    if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Choose another directory.")
    token = getpass.getpass("GitHub token (leave blank for a public repo): ").strip()
    git_env = os.environ.copy()
    git_env["GIT_TERMINAL_PROMPT"] = "0"
    if token:
        encoded = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        # Authentication is passed only to this subprocess, never saved in the URL.
        git_env.update(GIT_CONFIG_COUNT="2",
                       GIT_CONFIG_KEY_0="credential.helper", GIT_CONFIG_VALUE_0="",
                       GIT_CONFIG_KEY_1="http.https://github.com/.extraheader",
                       GIT_CONFIG_VALUE_1=f"Authorization: Basic {encoded}")
    if (REPO_DIR / ".git").is_dir():
        branch = subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip()
        if branch != BRANCH:
            raise RuntimeError(f"Existing checkout is on {branch!r}; expected {BRANCH!r}.")
        command = ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH]
    else:
        command = ["git", "clone", "--branch", BRANCH, REPO, str(REPO_DIR)]
    try:
        subprocess.run(command, env=git_env, check=True)
    except subprocess.CalledProcessError:
        raise RuntimeError("GitHub clone/pull failed. Check the repository URL, branch, and token access. "
                           "No later setup cells should be run until this succeeds.") from None
    os.chdir(REPO_DIR)
    print(f"Ready: {REPO_DIR} ({BRANCH})")

prepare_repository()


In [ ]:
# Keep data, weights and results on Drive
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/cs5284_project")

for name in ["data", "weights", "results"]:
    target = DRIVE_DIR / name
    target.mkdir(parents=True, exist_ok=True)
    link = Path(name)
    if not link.is_symlink():
        !rm -rf {link}
        link.symlink_to(target)
    print(f"{name}/ -> {target}")

In [ ]:
import sys
import subprocess
from pathlib import Path

def run_live(command):
    """Forward subprocess output into the notebook as it arrives."""
    import os
    child_env = dict(os.environ, PYTHONUNBUFFERED="1")
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env=child_env) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

STATE = "DE"  # Change to "MD" to run Maryland after Delaware.
TRAIN_PYTHON = "/content/cs5284-merged-env/bin/python"

run_live([sys.executable, "-m", "pip", "install", "-q", "uv"])
if not Path(TRAIN_PYTHON).exists():
    run_live([sys.executable, "-m", "uv", "venv", "--python", "3.12",
                    "--seed", "/content/cs5284-merged-env"])
run_live([TRAIN_PYTHON, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
run_live([TRAIN_PYTHON, "-c", "import torch; print('GPU:', torch.cuda.is_available()); "
                "assert torch.cuda.is_available(), 'Select a GPU runtime before training'"])

# Fail before expensive preprocessing if the selected checkout lacks Graph Transformer.
run_live([TRAIN_PYTHON, "-c", "from src.model.models import MODEL_NAMES; "
          "assert 'graph_transformer' in MODEL_NAMES, 'Push Graph Transformer and select its branch before training'; "
          "print('Available models:', MODEL_NAMES)"])


## Shared GPU Node2Vec-style embeddings

The merged implementation uses `baseline-fw`'s custom PyTorch skip-gram trainer: 128 dimensions, 20-step uniform walks (p=q=1), 10 walks per node, window 5 and one negative per context pair. This changes the paper's embedding training method; the settings and backend are recorded in JSON.

Embeddings are cached under `data/embeddings/node2vec_merged/`, preserving any earlier karateclub files. All four models use the same cached embeddings in the with-Node2Vec runs; the paired runs omit only those embeddings.

If Dataverse returns HTTP 403, download `DE.zip` (or `MD.zip`) from https://doi.org/10.7910/DVN/V71K5R and put it in `My Drive/cs5284_project/data/`, then rerun this cell. Existing data, weights and results do not need to be cleared.


In [ ]:
run_live([TRAIN_PYTHON, "-u", "-m", "src.data.node2vec", "--state", STATE])


In [ ]:
# Check embeddings using the project environment
embedding_check = """
import sys
# Sanity check: connected intersections should have more similar embeddings than random pairs
import numpy as np
import torch
from src.data.node2vec import node2vec_path

for state in [sys.argv[1]]:
    emb = np.load(node2vec_path(state))
    assert np.isfinite(emb).all(), "Non-finite embeddings"
    norms = np.linalg.norm(emb.astype(np.float64), axis=1, keepdims=True)
    assert (norms > 0).all(), "Zero embedding norm"
    emb = emb / norms
    src, dst = torch.load(f"data/{state}/adj_matrix.pt", weights_only=False).coalesce().indices().numpy()

    rng = np.random.default_rng(0)
    edges = rng.choice(len(src), 5000)
    others = rng.integers(0, len(emb), 5000)
    neighbour = (emb[src[edges]] * emb[dst[edges]]).sum(1).mean()
    random = (emb[src[edges]] * emb[others]).sum(1).mean()
    print(f"{state}: neighbour cosine {neighbour:.3f} vs random {random:.3f}")
    # Similarity is a diagnostic; downstream held-out metrics establish utility.
"""
run_live([TRAIN_PYTHON, "-c", embedding_check, STATE])


## Train the four-model comparison

MLP, GraphSAGE, GatedGCN and Graph Transformer use identical shared node/road features, data splits, losses, sampling and validation selection. Each runs with and without the same cached Node2Vec embeddings, for classification and regression: **16 configurations, 100 epochs, 3 seeds each (48 training runs per state)**.

Graph Transformer defaults: two blocks, width 256, four heads (64 channels/head), pre-LayerNorm, edge-modulated dot-product attention, and a 256 -> 512 -> 256 GELU feed-forward network with residual connections. It uses only existing directed edges, adds no self-loops, and keeps raw road attributes for the shared predictor. Road attributes are projected in each attention block; hidden edge states are not updated. It retains existing shared structural inputs rather than adding Laplacian PE. This is an adaptation of Week 09 attention, not a full reproduction of all tutorial labs. No PyG/DGL dependency is required.

GatedGCN retains its two-layer, width-256 residual gated encoder. Equal widths do not imply equal parameter counts across architectures.

To run only Graph Transformer, complete setup/embeddings and skip the six earlier model training cells. The two new cells add 12 seed runs, with distinct `<state>_graph_transformer_<task>_<n2v|no_n2v>` output directories. Re-running a configuration overwrites its outputs; incomplete training is not automatically resumed. No automatic tests or smoke training are run in this notebook.

Push the implementation first and set `BRANCH` in the clone cell to the branch containing it. Inspection/attention visualizations remain deferred.


### MLP: classification

Runs both Node2Vec settings across three seeds (6 training runs).


In [ ]:
SEEDS = ["0", "1", "2"]
EPOCHS = 100

model = "mlp"
task = "classification"
for use_node2vec in [False, True]:
    name = "with Node2Vec" if use_node2vec else "without Node2Vec"
    print(f"Starting {STATE} {model} {task}, {name}", flush=True)
    command = [TRAIN_PYTHON, "-u", "-m", "src.model.train", "--state", STATE,
               "--model", model, "--task", task, "--epochs", str(EPOCHS),
               "--seeds", *SEEDS, "--select-protocol", "all"]
    if not use_node2vec:
        command.append("--without-node2vec")
    run_live(command)


### MLP: regression

Runs both Node2Vec settings across three seeds (6 training runs).


In [ ]:
SEEDS = ["0", "1", "2"]
EPOCHS = 100

model = "mlp"
task = "regression"
for use_node2vec in [False, True]:
    name = "with Node2Vec" if use_node2vec else "without Node2Vec"
    print(f"Starting {STATE} {model} {task}, {name}", flush=True)
    command = [TRAIN_PYTHON, "-u", "-m", "src.model.train", "--state", STATE,
               "--model", model, "--task", task, "--epochs", str(EPOCHS),
               "--seeds", *SEEDS, "--select-protocol", "all"]
    if not use_node2vec:
        command.append("--without-node2vec")
    run_live(command)


### GraphSAGE: classification

Runs both Node2Vec settings across three seeds (6 training runs).


In [ ]:
SEEDS = ["0", "1", "2"]
EPOCHS = 100

model = "graphsage"
task = "classification"
for use_node2vec in [False, True]:
    name = "with Node2Vec" if use_node2vec else "without Node2Vec"
    print(f"Starting {STATE} {model} {task}, {name}", flush=True)
    command = [TRAIN_PYTHON, "-u", "-m", "src.model.train", "--state", STATE,
               "--model", model, "--task", task, "--epochs", str(EPOCHS),
               "--seeds", *SEEDS, "--select-protocol", "all"]
    if not use_node2vec:
        command.append("--without-node2vec")
    run_live(command)


### GraphSAGE: regression

Runs both Node2Vec settings across three seeds (6 training runs).


In [ ]:
SEEDS = ["0", "1", "2"]
EPOCHS = 100

model = "graphsage"
task = "regression"
for use_node2vec in [False, True]:
    name = "with Node2Vec" if use_node2vec else "without Node2Vec"
    print(f"Starting {STATE} {model} {task}, {name}", flush=True)
    command = [TRAIN_PYTHON, "-u", "-m", "src.model.train", "--state", STATE,
               "--model", model, "--task", task, "--epochs", str(EPOCHS),
               "--seeds", *SEEDS, "--select-protocol", "all"]
    if not use_node2vec:
        command.append("--without-node2vec")
    run_live(command)


### GatedGCN: classification

Runs both Node2Vec settings across three seeds (6 training runs).


In [ ]:
SEEDS = ["0", "1", "2"]
EPOCHS = 100

model = "gatedgcn"
task = "classification"
for use_node2vec in [False, True]:
    name = "with Node2Vec" if use_node2vec else "without Node2Vec"
    print(f"Starting {STATE} {model} {task}, {name}", flush=True)
    command = [TRAIN_PYTHON, "-u", "-m", "src.model.train", "--state", STATE,
               "--model", model, "--task", task, "--epochs", str(EPOCHS),
               "--seeds", *SEEDS, "--select-protocol", "all"]
    if not use_node2vec:
        command.append("--without-node2vec")
    run_live(command)


### GatedGCN: regression

Runs both Node2Vec settings across three seeds (6 training runs).


In [ ]:
SEEDS = ["0", "1", "2"]
EPOCHS = 100

model = "gatedgcn"
task = "regression"
for use_node2vec in [False, True]:
    name = "with Node2Vec" if use_node2vec else "without Node2Vec"
    print(f"Starting {STATE} {model} {task}, {name}", flush=True)
    command = [TRAIN_PYTHON, "-u", "-m", "src.model.train", "--state", STATE,
               "--model", model, "--task", task, "--epochs", str(EPOCHS),
               "--seeds", *SEEDS, "--select-protocol", "all"]
    if not use_node2vec:
        command.append("--without-node2vec")
    run_live(command)


### Graph Transformer: classification

Runs both Node2Vec settings across three seeds (6 training runs).


In [ ]:
SEEDS = ["0", "1", "2"]
EPOCHS = 100

NUM_HEADS = 4  # Must divide hidden channels (default 256).
model = "graph_transformer"
task = "classification"
for use_node2vec in [False, True]:
    name = "with Node2Vec" if use_node2vec else "without Node2Vec"
    print(f"Starting {STATE} {model} {task}, {name}", flush=True)
    command = [TRAIN_PYTHON, "-u", "-m", "src.model.train", "--state", STATE,
               "--model", model, "--task", task, "--epochs", str(EPOCHS),
               "--seeds", *SEEDS, "--select-protocol", "all", "--num-heads", str(NUM_HEADS)]
    if not use_node2vec:
        command.append("--without-node2vec")
    run_live(command)


### Graph Transformer: regression

Runs both Node2Vec settings across three seeds (6 training runs).


In [ ]:
SEEDS = ["0", "1", "2"]
EPOCHS = 100

NUM_HEADS = 4  # Must divide hidden channels (default 256).
model = "graph_transformer"
task = "regression"
for use_node2vec in [False, True]:
    name = "with Node2Vec" if use_node2vec else "without Node2Vec"
    print(f"Starting {STATE} {model} {task}, {name}", flush=True)
    command = [TRAIN_PYTHON, "-u", "-m", "src.model.train", "--state", STATE,
               "--model", model, "--task", task, "--epochs", str(EPOCHS),
               "--seeds", *SEEDS, "--select-protocol", "all", "--num-heads", str(NUM_HEADS)]
    if not use_node2vec:
        command.append("--without-node2vec")
    run_live(command)


## Results and published reference values

These merged runs use different features, target units and embedding training from the paper. Published scores are context, not an equivalent protocol. Only completed merged runs are shown.


In [ ]:
import json
import pandas as pd

# Paper Table 2 (test split, mean over 3 seeds)
PAPER = {
    ("DE", "mlp"): {"AUROC": 0.755, "MAE": 1.4},
    ("DE", "graphsage"): {"AUROC": 0.876, "MAE": 0.3},
    ("MD", "mlp"): {"AUROC": 0.741, "MAE": 0.4},
    ("MD", "graphsage"): {"AUROC": 0.876, "MAE": 0.4},
}

rows = []
for path in sorted(Path("results").glob("*/metrics.json")):
    result = json.load(open(path))
    cfg, test = result["config"], result["summary"]["test"]
    if "use_node2vec" not in cfg or not result.get("complete", False):
        continue
    metric = "AUROC" if cfg["task"] == "classification" else "MAE"
    row = {"run": cfg["run_name"], "state": cfg["state"], "model": cfg["model"], "node2vec": cfg["use_node2vec"], "metric": metric}
    for protocol in ["all", "balanced"]:
        m = test[protocol][metric]
        row[protocol] = f"{m['mean']:.3f} ± {m['std']:.3f}"
    row["paper"] = PAPER.get((cfg["state"], cfg["model"]), {}).get(metric)
    rows.append(row)

pd.DataFrame(rows)